In [4]:
system_prompt = """

You are a personal chef. The user will upload the image of the refrigerator for the list of ingredients they have left over.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [6]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [7]:
print(uploader.value)

({'name': 'FridgePicture_PersonalChefAgent.jpeg', 'type': 'image/jpeg', 'size': 80475, 'content': <memory at 0x00000245035C50C0>, 'last_modified': datetime.datetime(2026, 7, 8, 17, 20, 49, 673000, tzinfo=datetime.timezone.utc)},)


In [8]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about the ingredients and think of the recipes"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])



In [13]:
response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

Nice, with eggs and milk you have a lot of quick, comforting options. From the photo I see:
- Organic whole milk (half gallon)
- A carton of eggs (24, Grade AA)

Here are some tasty recipe ideas you can make with eggs and milk (and common pantry items). I’ve included simple links you can follow for full ingredients and step-by-step instructions. Tell me which you’d like and I can give you the full recipe.

Pancakes
- Why it’s good: Classic breakfast that uses eggs and milk as the base (plus flour, baking powder, sugar).
- Quick starter: beat eggs with milk, whisk into dry ingredients, cook on a hot pan.
- Sources for recipes: 
  - Favorite Pancakes (with eggs) – Food Hero: https://www.foodhero.org/recipes/favorite-pancakes-with-eggs
  - Homemade Pancakes – Mostly Homemade Mom: https://www.mostlyhomemademom.com/homemade-pancakes

French toast
- Why it’s good: Bread soaked in a creamy egg-milk mixture, then fried to golden perfection.
- Quick starter: whisk eggs with milk (plus a pinch o

In [14]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Tell me about the ingredients and think of the recipes '}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='da4d1d74-53f3-4863-9f3a-e10ee16dacc1'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 998, 'prompt_tokens': 1927, 'total_tokens': 2925, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzQJJ37uVbPc9FDXn6TG4sBufwMNl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f42c3-2d0d-72d1-a159-aaceab02d0b2-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes using 